- Generates and saves Spectral Plots for verified Rose-crested Blue Pipit audio files
- Saves spectral plot stats

In [ ]:
import pandas as pd
import os
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# load metadata
metadata = pd.read_excel("/content/drive/MyDrive/visual_analytics_project/updated_AllBirdsv4.xlsx")

In [ ]:
# preview
metadata.head()

,File ID,English_name,Vocalization_type,Quality,Time,Date,X,Y,Header
0,402254,Rose-crested Blue Pipit,call,no score,13:30,2018-02-08,49,63.0,Data
1,406171,Rose-crested Blue Pipit,call,A,7:48,2017-06-07,125,133.0,Data
2,405901,Rose-crested Blue Pipit,call,A,12:00,2018-02-08,58,76.0,Data
3,405548,Rose-crested Blue Pipit,song,A,11:00,2018-03-10,55,125.0,Data
4,401782,Rose-crested Blue Pipit,song,A,6:00,2008-06-29,129,123.0,Data


In [ ]:
# get only rose-crested blue pipit data
pipit_data = metadata[metadata['English_name'] == 'Rose-crested Blue Pipit']
pipit_data.head()

,File ID,English_name,Vocalization_type,Quality,Time,Date,X,Y,Header
0,402254,Rose-crested Blue Pipit,call,no score,13:30,2018-02-08,49,63.0,Data
1,406171,Rose-crested Blue Pipit,call,A,7:48,2017-06-07,125,133.0,Data
2,405901,Rose-crested Blue Pipit,call,A,12:00,2018-02-08,58,76.0,Data
3,405548,Rose-crested Blue Pipit,song,A,11:00,2018-03-10,55,125.0,Data
4,401782,Rose-crested Blue Pipit,song,A,6:00,2008-06-29,129,123.0,Data


In [ ]:
print(f"Total Pipit recordings: {len(pipit_data)}")

Total Pipit recordings: 186


In [ ]:
# access audio folder
all_birds = "/content/drive/MyDrive/visual_analytics_project/all_birds"

In [ ]:
# function to plot spectral data for a given audio file path
def plot_spectral(audio_path, title="Spectral Plot", save_path=None):
    #  load audio file
    y, sr = librosa.load(audio_path)

    # compute FFT: magnitude spectrum
    fft = np.fft.fft(y)
    magnitude = np.abs(fft)
    freq = np.fft.fftfreq(len(magnitude), 1/sr)

    # keep only the positive half of the spectrum
    left_freq = freq[:len(freq)//2]
    left_magnitude = magnitude[:len(magnitude)//2]

    # plot
    plt.figure(figsize=(10, 4))
    plt.plot(left_freq, left_magnitude)
    plt.title(title)
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Magnitude')
    plt.grid(True)
    plt.tight_layout()

    # save the plot to a file
    if save_path:
        plt.savefig(save_path)
        plt.close()  # close to avoid open figures
    else:
        plt.show()

In [ ]:
# specify output directory
output_dir = "/content/drive/MyDrive/visual_analytics_project/spectral_plots/pipit"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# list to save spectral data
spectral_data = []

In [ ]:
# process all rose-crested blue pipit data and generate spectral plots
for _, row in pipit_data.iterrows():
    # extract file ID
    filename = row['File ID']
    file_id = str(filename).split('-')[-1].split('.')[0]

    # file path
    file_path = os.path.join(all_birds, f"Rose-Crested-Blue-Pipit-{file_id}.mp3")

    # specify title and save path for plot
    title = f"Spectral Plot - Verified Pipit {file_id}"
    save_path = os.path.join(output_dir, f"{file_id}_spectral_plot.png")

    # plot and save
    plot_spectral(file_path, title, save_path)

    # add statistics to data list and extract
    y, sr = librosa.load(file_path)
    fft = np.fft.fft(y)
    magnitude = np.abs(fft)
    freq = np.fft.fftfreq(len(magnitude), 1/sr)

    left_freq = freq[:len(freq)//2]
    left_magnitude = magnitude[:len(magnitude)//2]

    peak_freq = left_freq[np.argmax(left_magnitude)]
    mean_freq = np.mean(left_freq)
    total_magnitude = np.sum(left_magnitude)
    entropy = -np.sum((left_magnitude / total_magnitude) * np.log2(left_magnitude / total_magnitude + 1e-10))

    spectral_data.append({
        'File ID': file_id,
        'Peak Frequency': peak_freq,
        'Mean Frequency': mean_freq,
        'Total Magnitude': total_magnitude,
        'Spectral Entropy': entropy
    })

In [ ]:
# save spectral stats
pd.DataFrame(spectral_data).to_csv("pipit_spectral_statistics.csv", index=False)